# Second case: the tier's question and Marketing's pushback, argued from the same numbers

**Week 1, Tuesday afternoon. The second case, in pairs, forty minutes.** One of you answers Marketing,
the other answers the head of Retail-Plus, from the same file, and together you write the reply.

> **Marketing comes back with a new deck.** "Retail-Plus members spend 7 percent more every time they
> order, so the tier is healthy and the answer is still acquisition. And Retail-Plus web orders fell
> hardest, 24 to 9, so this is the website team's problem, not the tier's and not the app's."
>
> The marketing lead, Kalpa Retail

> **The tier asks.** "Fine. Which of my members do I call first?"
>
> The head of Retail-Plus

> **Kavya's review of the escalated case.** "You found a branch that moved for a reason nobody
> guessed. Do the same with Marketing's two new numbers: find what each is made of before you agree
> or disagree."

This is the exercise twin: each placeholder is a lettered choice in the comment above it. Run it from the top; it stops at the first placeholder you have not filled, which is intended.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
ORDERS = kit.load_records("C2_W01_D02_orders_STUDENT.py")
SEGMENTS = ["Retail-Core", "Retail-Plus", "Business", "Student"]


def tree_for(rows):
    """The revenue tree's leaves for any list of orders, returned as a dictionary (chapter 3)."""
    revenue = 0
    seen = {}
    for order in rows:
        revenue += order["amount"]
        seen[order["customer_id"]] = True
    orders = len(rows)
    customers = len(seen)
    return {"revenue": revenue, "orders": orders, "customers": customers,
            "orders_per_customer": orders / customers if customers else 0,
            "revenue_per_order": revenue / orders if orders else 0}


def pct_change(before, after):
    """Chapter 5's fixed helper: the change every time, in the same type."""
    return round(100 * (after - before) / before, 1)

print(len(ORDERS), "orders loaded")

In [ ]:
kit.side_by_side(
    kit.ladder(["7 percent more per order: a healthy tier?", "Web fell hardest: the website?",
                "Who to call first", "The reply and the evidence"], show=False),
    kit.matrix(["Marketing", "the head of Retail-Plus"], ["claim", "test"],
               [["a healthy tier; the website", "the tier's own tree; web in another segment"], ["which members", "orders per member, Q1 to Q2"]],
               show=False),
)

## Part 1. "Members spend 7 percent more per order, so the tier is healthy"

Revenue per order is one leaf of the tree. Build the tier's tree for both quarters, then put the
leaves back together into the tier's revenue.

In [ ]:
plus = {q: [o for o in ORDERS if o["segment"] == "Retail-Plus" and o["quarter"] == q] for q in ("Q1", "Q2")}
t1, t2 = tree_for(plus["Q1"]), tree_for(plus["Q2"])
ratio = {leaf: t2[leaf] / t1[leaf] for leaf in ("customers", "orders_per_customer", "revenue_per_order")}
kit.table(["Leaf", "Q1", "Q2", "Q2 over Q1"],
          [("members", t1["customers"], t2["customers"], f'{ratio["customers"]:.3f}'),
           ("orders per member", f'{t1["orders_per_customer"]:.2f}', f'{t2["orders_per_customer"]:.2f}', f'{ratio["orders_per_customer"]:.3f}'),
           ("revenue per order", kit.rupees(round(t1["revenue_per_order"])), kit.rupees(round(t2["revenue_per_order"])),
            f'{ratio["revenue_per_order"]:.3f}')],
          caption="Retail-Plus, leaf by leaf")
# TODO 1. Which expression gives the tier's Q2 revenue as a share of its Q1 revenue, from the leaves?
#   a) ratio["customers"] * ratio["orders_per_customer"] * ratio["revenue_per_order"]
#   b) ratio["orders_per_customer"] + ratio["revenue_per_order"] - 1
#   c) ratio["revenue_per_order"]
#   d) (ratio["orders_per_customer"] + ratio["revenue_per_order"]) / 2
tier_share = __TODO1__
kit.columns(["Q1", "Q2, from your expression"], [("tier revenue", [t1["revenue"], round(t1["revenue"] * tier_share)])],
            fmt=kit.rupees, width=520, title="Retail-Plus revenue, Q1 and Q2")

In [ ]:
kit.check("your expression gives back the tier's Q2 revenue, counted order by order",
          abs(t1["revenue"] * tier_share - t2["revenue"]) < 1, kit.rupees(round(t1["revenue"] * tier_share)))

**Item 1 in your brief** goes in now, from what this part printed.

## Part 2. "Retail-Plus web orders fell hardest, so it is the website"

A broken website hurts every customer who uses it. If the website were the cause, what would another
segment's web orders show?

In [ ]:
web = {(o["segment"], o["quarter"]): 0 for o in ORDERS}
for o in ORDERS:
    if o["channel"] == "web":
        web[(o["segment"], o["quarter"])] += 1
# TODO 2. If the website were broken for everyone, which other segment's web orders would also have fallen hard?
#   a) "Retail-Plus"
#   b) "Student"
#   c) "Retail-Core"
#   d) "none"
also_falls = __TODO2__
picked = also_falls if also_falls in SEGMENTS else "Retail-Plus"
kit.columns(["Retail-Plus", picked], [("Q1 web orders", [web[("Retail-Plus", "Q1")], web[(picked, "Q1")]]),
                                      ("Q2 web orders", [web[("Retail-Plus", "Q2")], web[(picked, "Q2")]])],
            width=560, title="Web orders: the members against the segment you chose")

In [ ]:
comparison = (web[(also_falls, "Q1")], web[(also_falls, "Q2")]) if also_falls in SEGMENTS else (0, 0)
kit.check("the segment you drew is not the members, and had enough web orders in Q1 to show a fall",
          also_falls != "Retail-Plus" and comparison[0] >= 10, f"{comparison[0]} and {comparison[1]}")
kit.check("Retail-Plus web orders fell 24 to 9", (web[("Retail-Plus", "Q1")], web[("Retail-Plus", "Q2")]) == (24, 9))

**Item 2 in your brief** goes in now, from what this part printed.

## Part 3. "Which of my members do I call first?"

Count each member's orders in Q1 and Q2, and decide who goes first.

In [ ]:
per = {q: {} for q in ("Q1", "Q2")}
for o in ORDERS:
    if o["segment"] == "Retail-Plus":
        per[o["quarter"]][o["customer_id"]] = per[o["quarter"]].get(o["customer_id"], 0) + 1
pairs = {}
for cid in per["Q1"]:
    key = (per["Q1"][cid], per["Q2"].get(cid, 0))
    pairs[key] = pairs.get(key, 0) + 1
call_first = []
for cid in per["Q1"]:
    q1n, q2n = per["Q1"][cid], per["Q2"].get(cid, 0)
    # TODO 3. Which members go to the top of the call list?
    #   a) q2n == 0
    #   b) q1n - q2n >= 2
    #   c) q2n > q1n
    #   d) q1n == 1
    if __TODO3__:
        call_first.append(cid)
labels = [f"{a} to {b}" for a, b in sorted(pairs, reverse=True)]
kit.bars([(l, pairs[k]) for l, k in zip(labels, sorted(pairs, reverse=True))],
         title="Retail-Plus members, by orders in Q1 and in Q2")

In [ ]:
lost_by_list = sum(per["Q1"][c] - per["Q2"].get(c, 0) for c in call_first)
lost_by_tier = sum(per["Q1"].values()) - sum(per["Q2"].values())
kit.check("the members on your list carry more than half of the tier's lost orders", 2 * lost_by_list > lost_by_tier,
          f"{lost_by_list} of {lost_by_tier}")
kit.check("the per-member counts add back to the tier's orders", sum(per["Q1"].values()) + sum(per["Q2"].values()) == sum(1 for o in ORDERS if o["segment"] == "Retail-Plus"))

**Item 3 in your brief** goes in now, from what this part printed.

## Part 4. The reply, and the one request that tests the most

Item 4 in your brief goes in once this part has run.

In [ ]:
EVIDENCE = {
    "campaigns": "Marketing's new-member sign-ups by month from July",
    "export": "this export again, cut by city, channel and week from July",
    "app_logs": "the app's reorder logs by week since the 25 August release",
    "tier_log": "the tier's July change log, renewals and support tickets",
}
# TODO 4. The tier lost 25 orders and chapter 6 capped the button at about 4 of them. Which request goes first?
#   a) "campaigns"
#   b) "export"
#   c) "app_logs"
#   d) "tier_log"
first_request = __TODO4__
kit.tree({"label": "the reply", "kind": "lit", "branches": [
    ("to Marketing", {"label": f"tier revenue x {tier_share:.2f}\nweb tested on {also_falls}", "kind": "known"}),
    ("to the tier", {"label": f"call the {len(call_first)}\nat the top of the list", "kind": "known"}),
    ("first request", {"label": EVIDENCE[first_request][:34], "kind": "unknown"})]},
    title="The pair's reply, in three branches")
import hashlib
kit.check("your pick matches the key, stored as a fingerprint", hashlib.sha256(first_request.encode()).hexdigest()[:10] == "a49850fe17")
kit.check("your pick names one of the four sources", first_request in EVIDENCE)

In [ ]:
kit.check_summary()